In [ ]:
print("Installing libraries...")

!pip install -q sentence-transformers langchain-chroma langchain-community google-genai pypdf gradio


In [ ]:
from google.colab import userdata  # keys are stored in Colab Secrets
# ==========================================
#  RAMADAN DIABETES EXPERT - CLEAN VERSION
# ==========================================

# --- IMPORTS ---
import gradio as gr
from google import genai
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from google.colab import drive
import os

# --- SETUP ---
drive.mount('/content/drive')

# --- EMBEDDING CLASS ---
class SentenceTransformerEmbeddings:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

# --- LOAD DATABASE ---
db_path = "/content/drive/MyDrive/Ramadan_RAG_Data/ramadan_db_v0"
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

print("🧠 Loading medical database...")
try:
    embeddings = SentenceTransformerEmbeddings()
    db = Chroma(persist_directory=db_path, embedding_function=embeddings)
    count = db._collection.count()
    print(f" Database loaded: {count} medical chunks ready")
except Exception as e:
    print(f" Database error: {e}")

# --- CORE FUNCTION ---
def ramadan_expert(message, history):
    try:
        # 1. Search medical database
        docs = db.similarity_search(message, k=4)
        context = "\n\n".join([d.page_content for d in docs])

        # 2. Build conversation history
        conversation = ""
        for human_msg, ai_msg in history:
            conversation += f"Patient: {human_msg}\nExpert: {ai_msg}\n\n"

        # 3. Build prompt
        prompt = f"""You are a professional Ramadan Diabetes Expert with deep knowledge of Islamic fasting and diabetes management.
Your role is to help diabetic patients understand if they can safely fast during Ramadan, based on international medical guidelines.
Do NOT mention any specific hospital, clinic, doctor name, or location.
Keep all advice general and based on international diabetes guidelines.

IMPORTANT RULES:
- Detect the language the patient is using (Moroccan Darija, Arabic,Amazigh, French, or English)
- ALWAYS respond in the SAME language the patient used
- Be warm, professional, and caring
- Never diagnose - only advise based on medical guidelines

MEDICAL GUIDELINES FROM DATABASE:
{context}

CONVERSATION HISTORY:
{conversation}

PATIENT MESSAGE: {message}

FASTING RISK ASSESSMENT:
HIGH RISK (say fasting is FORBIDDEN):
- HbA1c > 9%
- Blood sugar < 70 mg/dL (hypoglycemia)
- Blood sugar > 300 mg/dL
- Type 1 diabetes (generally high risk)
- Recent severe complications

MODERATE RISK (fasting allowed WITH strict monitoring):
- HbA1c 7-9%
- Well controlled Type 2 diabetes
- Taking insulin or sulfonylureas

LOW RISK (fasting generally safe):
- HbA1c < 7%
- Diet controlled or Metformin only
- No recent complications

If information is missing, ASK the patient for:
- Their HbA1c (Khazan) level
- Their current blood sugar (Sskkar)
- Their diabetes type and medications

RESPONSE:"""

        # 4. Call Gemini
        response = client.models.generate_content(
            model='models/gemini-2.5-flash',
            contents=prompt
        )
        return response.text

    except Exception as e:
        return f" System error: {e}"

# --- PROFESSIONAL UI ---
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="gray",
    ),
    css="""
        .header-box {
            background: linear-gradient(135deg, #1a5276, #2980b9);
            padding: 25px;
            border-radius: 12px;
            text-align: center;
            margin-bottom: 20px;
            color: white;
        }
        .info-box {
            background: #f0f4f8;
            border-left: 4px solid #2980b9;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 15px;
        }
        .disclaimer {
            background: #fef9e7;
            border-left: 4px solid #f39c12;
            padding: 12px;
            border-radius: 8px;
            font-size: 13px;
        }
        footer { display: none; }
    """
) as demo:

    # Header
    gr.HTML("""
        <div class="header-box">
            <h1 style="margin:0; font-size:28px;"> Ramadan Diabetes Expert</h1>
            <p style="margin:8px 0 0 0; font-size:15px; opacity:0.9;">
                Powered by IDF Ramadan Diabetes Guidelines
            </p>
            <p style="margin:4px 0 0 0; font-size:13px; opacity:0.7;">
                Based on IDF-DAR Practical Guidelines (2022)
            </p>
        </div>
    """)

    with gr.Row():
        # Left column - Chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Medical Consultation",
                height=450,
                bubble_full_width=False,
                avatar_images=(
                    None,
                    "https://img.icons8.com/color/96/doctor-male.png"
                )
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Type your question in any language (Darija, Arabic,Amazigh French, English)...",
                    label="Your Question",
                    scale=4,
                    lines=2
                )
                send_btn = gr.Button(
                    "Send 📨",
                    variant="primary",
                    scale=1
                )

            gr.ClearButton([msg_input, chatbot], value="🗑️ Clear Chat")

        # Right column - Info panel
        with gr.Column(scale=1):
            gr.HTML("""
                <div class="info-box">
                    <h3 style="margin:0 0 10px 0; color:#1a5276;">⚠️ Risk Levels</h3>
                    <p style="color:#e74c3c;">🔴 <b>HIGH RISK</b><br>
                    HbA1c > 9%<br>
                    Sugar < 70 or > 300<br>
                    Type 1 Diabetes</p>
                    <p style="color:#f39c12;">🟡 <b>MODERATE</b><br>
                    HbA1c 7-9%<br>
                    Controlled Type 2</p>
                    <p style="color:#27ae60;">🟢 <b>LOW RISK</b><br>
                    HbA1c < 7%<br>
                    Diet controlled</p>
                </div>

                <div class="info-box">
                    <h3 style="margin:0 0 10px 0; color:#1a5276;">💬 Try Asking</h3>
                    <p style="font-size:13px;">
                    🇲🇦 "Sskkar diali 60, wach nsom?"<br><br>
                    🇫🇷 "Mon HbA1c est 10%, puis-je jeûner?"<br><br>
                    🇬🇧 "I have Type 2, can I fast?"<br><br>
                    🇸🇦 "هل يمكنني الصيام مع السكري؟"
                    </p>
                </div>

                <div class="disclaimer">
                    <b>⚕️ Medical Disclaimer</b><br>
                    This tool provides general guidance based on IDF-DAR guidelines.
                    Always consult your doctor before making fasting decisions.
                </div>
            """)

    # Example questions
    gr.Examples(
        examples=[
            "Sskkar diali 60, wach nsom?",
            "Khazan diali 10%, wach nsom?",
            "Mon HbA1c est 7.5%, puis-je jeûner?",
            "I have Type 1 diabetes, can I fast?",
            "هل يمكنني الصيام وأنا أتناول الأنسولين؟",
            "Wach metformin momkin nakhdo w ana sayem?"
        ],
        inputs=msg_input,
        label="📋 Example Questions"
    )

    # Event handlers
    def respond(message, chat_history):
        if not message.strip():
            return "", chat_history
        bot_response = ramadan_expert(message, chat_history)
        chat_history.append((message, bot_response))
        return "", chat_history

    send_btn.click(
        respond,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

    msg_input.submit(
        respond,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot]
    )

# --- LAUNCH ---
print("\n Launching Ramadan Diabetes Expert...")
print("=" * 60)
demo.launch(
    share=True,
    debug=False,
    show_error=True
)